<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [ ]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260723", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [ ]:
robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
# robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture change
</span>

In [ ]:
desired_pos = ysw.DEMO_BASIC_POS_260826
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Position Control
</span>

In [ ]:
T_SC_target_before_open_gripper_blackObj = 

pos_before_open_gripper_blackObj = T_SC_target_before_open_gripper_blackObj[:3,3]
ori_before_open_gripper_blackObj = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper_blackObj[:3,:3])       

if (grasp_arm_blackObj == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper_blackObj,                  
                                                "rpy": ori_before_open_gripper_blackObj,                    
                                            },                 
                                        ]
    
elif (grasp_arm_blackObj == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "base",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper_blackObj,                  
                                                "rpy": ori_before_open_gripper_blackObj,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    minimum_time=position_control_minimum_time,
    # minimum_time=position_control_minimum_time * minimum_time_scaler,
    control_hold_time=0.5,
    get_return=True
)